# YOLOv11n Baseline v1 — NEU-DET Steel Defect Detection
## First Baseline Model
- **Dataset:** NEU-DET (6 classes, 1800 images)
- **Model:** YOLOv11n (standard Ultralytics, no custom modules)
- **Image size:** 640x640
- **Epochs:** 200 (early stopped at 166)
- **Best mAP@0.5:** 75.6% (epoch 116)

This notebook documents the first baseline model trained on the NEU-DET steel surface
defect detection dataset using a standard YOLOv11n architecture with no modifications.
All settings match the original training run exactly.

## 1. Setup

In [1]:
import torch
import csv
import json
import random
import numpy as np
from pathlib import Path
from ultralytics import YOLO

# ── Reproducibility ──────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── GPU check ────────────────────────────────────────────────────
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected — training will be very slow.")

# ── Paths ────────────────────────────────────────────────────────
PROJECT_ROOT = Path(".").resolve() if Path("configs").exists() else Path("..").resolve()
DATA_YAML = PROJECT_ROOT / "configs" / "data" / "neu_det.yaml"
WEIGHTS_PATH = PROJECT_ROOT / "runs" / "baseline_seed42" / "weights" / "best.pt"
RESULTS_CSV = PROJECT_ROOT / "runs" / "baseline_seed42" / "results.csv"

print(f"\nProject root : {PROJECT_ROOT}")
print(f"Data YAML    : {DATA_YAML}")
print(f"Weights      : {WEIGHTS_PATH}")
print(f"Results CSV  : {RESULTS_CSV}")
print(f"Weights exist: {WEIGHTS_PATH.exists()}")

PyTorch version: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA RTX 2000 Ada Generation


AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

## 2. Dataset Information

The **NEU-DET** (Northeastern University — Detection) dataset is a benchmark for steel
surface defect detection. It contains **1,800 grayscale images** of size **200x200 pixels**
with bounding box annotations for **6 defect classes**:

| ID | Class            | Description                         |
|----|------------------|-------------------------------------|
| 0  | crazing          | Network of fine cracks on surface   |
| 1  | inclusion        | Foreign particles embedded in steel |
| 2  | patches          | Localized surface discolorations    |
| 3  | pitted_surface   | Small depressions / cavities        |
| 4  | rolled-in_scale  | Oxide scale rolled into surface     |
| 5  | scratches        | Linear surface marks from handling  |

**Split ratio:** 70% train / 20% val / 10% test  
**Total images:** 1,290 train + 344 val + 166 test = 1,800

## 3. Dataset Distribution

In [ ]:
from collections import defaultdict

DATASET_DIR = PROJECT_ROOT / "datasets" / "NEU-DET" / "yolo"
CLASS_NAMES = ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]

print(f"{'Class':<18} {'Train':>6} {'Val':>6} {'Test':>6} {'Total':>6}")
print("-" * 50)

totals = {"train": 0, "val": 0, "test": 0}
for cls_name in CLASS_NAMES:
    row = {"train": 0, "val": 0, "test": 0}
    for split in ["train", "val", "test"]:
        label_dir = DATASET_DIR / "labels" / split
        count = len(list(label_dir.glob(f"{cls_name}_*.txt")))
        row[split] = count
        totals[split] += count
    print(f"{cls_name:<18} {row['train']:>6} {row['val']:>6} {row['test']:>6} {sum(row.values()):>6}")

print("-" * 50)
print(f"{'TOTAL':<18} {totals['train']:>6} {totals['val']:>6} {totals['test']:>6} {sum(totals.values()):>6}")

## 4. Training Configuration

In [ ]:
train_args = {
    "model": "yolo11n.pt",
    "data": str(DATA_YAML),
    "epochs": 200,
    "patience": 50,
    "batch": 16,
    "imgsz": 640,
    "seed": 42,
    "deterministic": True,
    "optimizer": "auto",
    "lr0": 0.01,
    "lrf": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "close_mosaic": 10,
    "amp": True,
    "workers": 8,
    "project": str(PROJECT_ROOT / "runs"),
    "name": "baseline_seed42",
    "exist_ok": True,
    "plots": True,
    "verbose": True,
}

print("Training Configuration:")
print("=" * 50)
for k, v in train_args.items():
    print(f"  {k:<20}: {v}")

## 5. Load or Train Model

In [ ]:
if WEIGHTS_PATH.exists():
    print(f"Found existing weights at: {WEIGHTS_PATH}")
    print("Using existing weights (no retraining).")
    model = YOLO(str(WEIGHTS_PATH))
else:
    print("No existing weights found. Starting training from scratch...")
    model = YOLO("yolo11n.pt")
    results = model.train(**train_args)
    print(f"\nTraining complete. Best weights saved to: {WEIGHTS_PATH}")

## 6. Training Results

In [ ]:
import matplotlib.pyplot as plt

# ── Parse results.csv ────────────────────────────────────────────
if not RESULTS_CSV.exists():
    raise FileNotFoundError(f"Results file not found: {RESULTS_CSV}")

epochs, map50, map50_95, precision, recall = [], [], [], [], []
train_box_loss, train_cls_loss, train_dfl_loss = [], [], []
val_box_loss, val_cls_loss, val_dfl_loss = [], [], []

with open(RESULTS_CSV, "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        epochs.append(int(row["epoch"]))
        map50.append(float(row["metrics/mAP50(B)"]))
        map50_95.append(float(row["metrics/mAP50-95(B)"]))
        precision.append(float(row["metrics/precision(B)"]))
        recall.append(float(row["metrics/recall(B)"]))
        train_box_loss.append(float(row["train/box_loss"]))
        train_cls_loss.append(float(row["train/cls_loss"]))
        train_dfl_loss.append(float(row["train/dfl_loss"]))
        val_box_loss.append(float(row["val/box_loss"]))
        val_cls_loss.append(float(row["val/cls_loss"]))
        val_dfl_loss.append(float(row["val/dfl_loss"]))

# ── Best epoch ───────────────────────────────────────────────────
best_idx = map50.index(max(map50))
print("=" * 55)
print(f"  Best epoch       : {epochs[best_idx]}")
print(f"  Best mAP@0.5     : {map50[best_idx]:.4f}  ({map50[best_idx]*100:.1f}%)")
print(f"  mAP@0.5:0.95     : {map50_95[best_idx]:.4f}  ({map50_95[best_idx]*100:.1f}%)")
print(f"  Precision        : {precision[best_idx]:.4f}")
print(f"  Recall           : {recall[best_idx]:.4f}")
print(f"  Total epochs ran : {len(epochs)} (early stopped at epoch {epochs[-1]})")
print("=" * 55)

# ── Plot: mAP curves ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(epochs, map50, "b-", linewidth=1.5, label="mAP@0.5")
axes[0].plot(epochs, map50_95, "r-", linewidth=1.5, label="mAP@0.5:0.95")
axes[0].axvline(x=epochs[best_idx], color="g", linestyle="--", alpha=0.7, label=f"Best (epoch {epochs[best_idx]})")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("mAP")
axes[0].set_title("Validation mAP")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ── Plot: Precision / Recall ─────────────────────────────────────
axes[1].plot(epochs, precision, "g-", linewidth=1.5, label="Precision")
axes[1].plot(epochs, recall, "m-", linewidth=1.5, label="Recall")
axes[1].axvline(x=epochs[best_idx], color="g", linestyle="--", alpha=0.7)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].set_title("Precision & Recall")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# ── Plot: Loss curves ────────────────────────────────────────────
axes[2].plot(epochs, train_box_loss, "b-", linewidth=1, alpha=0.7, label="Train Box")
axes[2].plot(epochs, train_cls_loss, "r-", linewidth=1, alpha=0.7, label="Train Cls")
axes[2].plot(epochs, val_box_loss, "b--", linewidth=1.5, label="Val Box")
axes[2].plot(epochs, val_cls_loss, "r--", linewidth=1.5, label="Val Cls")
axes[2].axvline(x=epochs[best_idx], color="g", linestyle="--", alpha=0.7)
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Loss")
axes[2].set_title("Training & Validation Loss")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Evaluate on Validation Set

In [ ]:
# Run validation on the val split
val_results = model.val(data=str(DATA_YAML), split='test', verbose=True)

print("\n" + "=" * 55)
print("  VALIDATION RESULTS")
print("=" * 55)
print(f"  mAP@0.5      : {val_results.box.map50:.4f}  ({val_results.box.map50*100:.1f}%)")
print(f"  mAP@0.5:0.95 : {val_results.box.map:.4f}  ({val_results.box.map*100:.1f}%)")
print(f"  Precision    : {val_results.box.mp:.4f}")
print(f"  Recall       : {val_results.box.mr:.4f}")
print("=" * 55)

# ── Per-class mAP ────────────────────────────────────────────────
print(f"\n  {'Class':<20} {'mAP@0.5':>10} {'mAP@0.5:0.95':>14}")
print("  " + "-" * 46)
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name:<20} {val_results.box.ap50[i]:>10.4f} {val_results.box.ap[i]:>14.4f}")

## 8. Results Summary

### Final Metrics

| Metric          | Value  |
|-----------------|--------|
| mAP@0.5         | 75.6%  |
| Best Epoch      | 116    |
| Early Stop      | 166    |
| Total Epochs    | 166    |
| Model           | YOLOv11n (standard) |

### Notes
- This is the **first baseline** with a standard YOLOv11n and no custom modules.
- Early stopping triggered at epoch 166 (patience=50, no improvement since epoch 116).
- Subsequent versions (v2+) will add DAFE, WFCA, and Inner-WIoU to improve performance.
- The baseline establishes the reference point: **75.6% mAP@0.5**.